# Synthetic IoT Data Generator for Dashboard

Notebook này sinh dữ liệu ảo phù hợp với:
- file `sensor_simulator.py`
- schema bảng đích sau smart filtering

Mục tiêu:
- 5 sensor, mỗi sensor đúng 1 loại metric
- mỗi sensor ít nhất 2000 dòng
- dữ liệu rải đều trong 24 giờ và phủ 7 ngày gần nhất
- ghi thẳng vào bảng đích Delta

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timedelta, timezone
import math
import random
from zoneinfo import ZoneInfo
# ===== Config =====
DB_NAME = "iot_analytics"
TARGET_TABLE = f"{DB_NAME}.smart_filtered_measurements"

ROWS_PER_SENSOR = 2400   # >= 2000
DAYS_BACK = 7
SEED = 42
VN_TZ = ZoneInfo("Asia/Ho_Chi_Minh")

# append | overwrite
WRITE_MODE = "append"

random.seed(SEED)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
  event_ts TIMESTAMP,
  sensor_id STRING,
  location STRING,
  metric_type STRING,
  metric_value DOUBLE,
  unit STRING
) USING DELTA
PARTITIONED BY (metric_type)
''')

print("Target table:", TARGET_TABLE)
print("Rows per sensor:", ROWS_PER_SENSOR)
print("Days back:", DAYS_BACK)
print("Write mode:", WRITE_MODE)

Target table: iot_analytics.smart_filtered_measurements
Rows per sensor: 2400
Days back: 7
Write mode: append


In [0]:
# ===== Sensor profiles based on sensor_simulator.py =====
SENSOR_PROFILES = [
    {
        "sensor_id": "sensor_1",
        "location": "Living_Room",
        "metric_key": "temperature",
        "unit": "°C",
        "minimum": 15.0,
        "maximum": 40.0,
        "start": 24.5,
        "max_step": 0.45,
        "pull_strength": 0.24,
        "decimals": 2,
        "bias": 0.0,
    },
    {
        "sensor_id": "sensor_2",
        "location": "Living_Room",
        "metric_key": "humidity",
        "unit": "%",
        "minimum": 20.0,
        "maximum": 95.0,
        "start": 62.0,
        "max_step": 1.1,
        "pull_strength": 0.18,
        "decimals": 2,
        "bias": 0.0,
    },
    {
        "sensor_id": "sensor_3",
        "location": "Garden",
        "metric_key": "soil_moisture",
        "unit": "%",
        "minimum": 8.0,
        "maximum": 95.0,
        "start": 55.0,
        "max_step": 0.55,
        "pull_strength": 0.50,
        "decimals": 2,
        "irrigation_chance": 0.03,
    },
    {
        "sensor_id": "sensor_4",
        "location": "Outdoor",
        "metric_key": "light_intensity",
        "unit": "lux",
        "minimum": 0.0,
        "maximum": 60000.0,
        "start": 10.0,
        "max_step": 80.0,
        "pull_strength": 0.30,
        "decimals": 0,
        "light_peak": 38000.0,
        "light_night": 3.0,
    },
    {
        "sensor_id": "sensor_5",
        "location": "Outdoor",
        "metric_key": "pressure",
        "unit": "hPa",
        "minimum": 990.0,
        "maximum": 1035.0,
        "start": 1012.0,
        "max_step": 0.25,
        "pull_strength": 0.16,
        "decimals": 2,
        "bias": 0.0,
    },
]

In [0]:
# ===== Helper functions aligned with sensor_simulator.py =====
def clamp(value, minimum, maximum):
    return max(minimum, min(value, maximum))

def hour_fraction(dt):
    return dt.hour + dt.minute / 60 + dt.second / 3600

def bounded_random_walk(current, minimum, maximum, target, max_step, pull_strength):
    noise = random.uniform(-max_step, max_step)
    drift = (target - current) * pull_strength
    return clamp(current + noise + drift, minimum, maximum)

def target_for_metric(profile, current, now):
    metric_key = profile["metric_key"]
    hour = hour_fraction(now)

    temp_cycle = math.sin((2 * math.pi * (hour - 8)) / 24)
    daylight_factor = max(0.0, math.sin(math.pi * (hour - 6) / 12))
    pressure_cycle = math.sin((2 * math.pi * (hour - 3)) / 24)

    if metric_key == "temperature":
        return 24.5 + profile.get("bias", 0.0) + 5.5 * temp_cycle

    if metric_key == "humidity":
        return 62.0 + profile.get("bias", 0.0) - 12.0 * temp_cycle

    if metric_key == "soil_moisture":
        evap = 0.05 + 0.10 * daylight_factor + max(temp_cycle, 0.0) * 0.08
        target = current - evap
        if random.random() < profile.get("irrigation_chance", 0.03):
            target += random.uniform(4.0, 9.0)
        return target

    if metric_key == "light_intensity":
        light_night = profile.get("light_night", 3.0)
        light_peak = profile.get("light_peak", 38000.0)
        return light_night + daylight_factor * (light_peak - light_night)

    if metric_key == "pressure":
        return 1012.0 + profile.get("bias", 0.0) + 1.2 * pressure_cycle

    return current

def update_sensor_value(current, profile, now):
    target = target_for_metric(profile, current, now)
    return bounded_random_walk(
        current=current,
        minimum=profile["minimum"],
        maximum=profile["maximum"],
        target=target,
        max_step=profile["max_step"],
        pull_strength=profile["pull_strength"],
    )

def build_timestamps(days_back=7, rows_per_sensor=2400):
    # Tạo timestamps phủ 7 ngày gần nhất, rải đều 24h nhưng có jitter ngẫu nhiên
    end_time = datetime.now(VN_TZ).replace(microsecond=0)
    start_time = end_time - timedelta(days=days_back)

    total_seconds = int((end_time - start_time).total_seconds())
    step = total_seconds / rows_per_sensor

    ts = []
    for i in range(rows_per_sensor):
        base = start_time + timedelta(seconds=i * step)
        jitter_seconds = random.randint(-300, 300)  # lệch tối đa 5 phút
        t = base + timedelta(seconds=jitter_seconds)
        if t < start_time:
            t = start_time
        if t > end_time:
            t = end_time
        ts.append(t)

    ts = sorted(ts)
    return ts

In [0]:
# ===== Generate synthetic rows =====
rows = []

for profile in SENSOR_PROFILES:
    current_value = profile["start"]
    timestamps = build_timestamps(days_back=DAYS_BACK, rows_per_sensor=ROWS_PER_SENSOR)

    for ts in timestamps:
        current_value = update_sensor_value(current_value, profile, ts)
        decimals = profile["decimals"]
        measured_value = int(round(current_value)) if decimals == 0 else round(current_value, decimals)

        rows.append((
            ts,
            profile["sensor_id"],
            profile["location"],
            profile["metric_key"],
            float(measured_value),
            profile["unit"],
        ))

schema = T.StructType([
    T.StructField("event_ts", T.TimestampType(), False),
    T.StructField("sensor_id", T.StringType(), False),
    T.StructField("location", T.StringType(), True),
    T.StructField("metric_type", T.StringType(), False),
    T.StructField("metric_value", T.DoubleType(), False),
    T.StructField("unit", T.StringType(), True),
])

synthetic_df = spark.createDataFrame(rows, schema=schema)

print("Total rows generated:", synthetic_df.count())
display(synthetic_df.orderBy("event_ts"))

Total rows generated: 12000


event_ts,sensor_id,location,metric_type,metric_value,unit
2026-04-10T16:28:08.000+07:00,sensor_3,Garden,soil_moisture,54.48,%
2026-04-10T16:28:08.000+07:00,sensor_2,Living_Room,humidity,59.44,%
2026-04-10T16:28:08.000+07:00,sensor_4,Outdoor,light_intensity,7582.0,lux
2026-04-10T16:28:08.000+07:00,sensor_1,Living_Room,temperature,25.73,°C
2026-04-10T16:28:08.000+07:00,sensor_3,Garden,soil_moisture,54.19,%
2026-04-10T16:28:08.000+07:00,sensor_4,Outdoor,light_intensity,4388.0,lux
2026-04-10T16:29:29.000+07:00,sensor_5,Outdoor,pressure,1011.82,hPa
2026-04-10T16:29:42.000+07:00,sensor_1,Living_Room,temperature,26.67,°C
2026-04-10T16:31:17.000+07:00,sensor_5,Outdoor,pressure,1011.82,hPa
2026-04-10T16:31:34.000+07:00,sensor_2,Living_Room,humidity,58.66,%


In [0]:
# ===== Optional de-dup / sort before write =====
final_df = (
    synthetic_df
    .dropDuplicates(["event_ts", "sensor_id", "metric_type"])
)

print("Rows after dedup:", final_df.count())
display(
    final_df.groupBy("sensor_id", "metric_type")
    .agg(
        F.count("*").alias("rows"),
        F.min("event_ts").alias("min_ts"),
        F.max("event_ts").alias("max_ts")
    )
    .orderBy("sensor_id")
)

Rows after dedup: 11984


sensor_id,metric_type,rows,min_ts,max_ts
sensor_1,temperature,2397,2026-04-10T16:28:08.000+07:00,2026-04-17T16:23:04.000+07:00
sensor_2,humidity,2397,2026-04-10T16:28:08.000+07:00,2026-04-17T16:27:22.000+07:00
sensor_3,soil_moisture,2393,2026-04-10T16:28:08.000+07:00,2026-04-17T16:24:40.000+07:00
sensor_4,light_intensity,2397,2026-04-10T16:28:08.000+07:00,2026-04-17T16:26:27.000+07:00
sensor_5,pressure,2400,2026-04-10T16:29:29.000+07:00,2026-04-17T16:19:04.000+07:00


In [0]:
# ===== Write directly to target Delta table =====
(final_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .saveAsTable(TARGET_TABLE)
)

print(f"Done writing to {TARGET_TABLE} with mode={WRITE_MODE}")

Done writing to iot_analytics.smart_filtered_measurements with mode=append


In [0]:
# ===== Validation queries =====
display(
    spark.sql(f'''
        SELECT
            sensor_id,
            metric_type,
            COUNT(*) AS rows,
            MIN(event_ts) AS min_event_ts,
            MAX(event_ts) AS max_event_ts
        FROM {TARGET_TABLE}
        GROUP BY sensor_id, metric_type
        ORDER BY sensor_id
    ''')
)

display(
    spark.sql(f'''
        SELECT
            metric_type,
            COUNT(*) AS total_rows,
            ROUND(MIN(metric_value), 2) AS min_value,
            ROUND(MAX(metric_value), 2) AS max_value
        FROM {TARGET_TABLE}
        GROUP BY metric_type
        ORDER BY metric_type
    ''')
)

display(
    spark.sql(f'''
        SELECT *
        FROM {TARGET_TABLE}
        ORDER BY event_ts DESC
        LIMIT 100
    ''')
)

sensor_id,metric_type,rows,min_event_ts,max_event_ts
sensor_1,temperature,2397,2026-04-10T16:28:08.000+07:00,2026-04-17T16:23:04.000+07:00
sensor_2,humidity,2397,2026-04-10T16:28:08.000+07:00,2026-04-17T16:27:22.000+07:00
sensor_3,soil_moisture,2393,2026-04-10T16:28:08.000+07:00,2026-04-17T16:24:40.000+07:00
sensor_4,light_intensity,2397,2026-04-10T16:28:08.000+07:00,2026-04-17T16:26:27.000+07:00
sensor_5,pressure,2400,2026-04-10T16:29:29.000+07:00,2026-04-17T16:19:04.000+07:00


metric_type,total_rows,min_value,max_value
humidity,2397,48.03,76.35
light_intensity,2397,0.0,38046.0
pressure,2400,1009.9,1013.99
soil_moisture,2393,41.3,95.0
temperature,2397,18.16,30.89


event_ts,sensor_id,location,metric_type,metric_value,unit,threshold,store_reason,kafka_topic,kafka_partition,kafka_offset,processed_at
2026-04-17T16:27:22.000+07:00,sensor_2,Living_Room,humidity,51.57,%,null,null,null,null,null,null
2026-04-17T16:26:27.000+07:00,sensor_4,Outdoor,light_intensity,16616.0,lux,null,null,null,null,null,null
2026-04-17T16:24:40.000+07:00,sensor_3,Garden,soil_moisture,85.99,%,null,null,null,null,null,null
2026-04-17T16:23:04.000+07:00,sensor_1,Living_Room,temperature,29.14,°C,null,null,null,null,null,null
2026-04-17T16:22:54.000+07:00,sensor_1,Living_Room,temperature,29.6,°C,null,null,null,null,null,null
2026-04-17T16:22:39.000+07:00,sensor_4,Outdoor,light_intensity,17363.0,lux,null,null,null,null,null,null
2026-04-17T16:22:28.000+07:00,sensor_2,Living_Room,humidity,52.53,%,null,null,null,null,null,null
2026-04-17T16:20:26.000+07:00,sensor_3,Garden,soil_moisture,86.47,%,null,null,null,null,null,null
2026-04-17T16:19:04.000+07:00,sensor_5,Outdoor,pressure,1011.29,hPa,null,null,null,null,null,null
2026-04-17T16:18:38.000+07:00,sensor_3,Garden,soil_moisture,86.98,%,null,null,null,null,null,null
